# Run the VLM for the demo

One Colab session. Produces `vlm_outputs.json`, which you merge into the demo
cache locally so the demo can show a genuine **rules vs model** comparison.

**Runtime > Change runtime type > T4 GPU** before running anything.

What this does:
1. Installs the VLM dependencies.
2. Uploads the demo cache images + prompts from your machine.
3. Runs Qwen2-VL-2B on each one.
4. Downloads `vlm_outputs.json`.

It does *not* score anything. Scoring, coercion and validation all happen
locally with the same code the rules baseline goes through, so a difference on
screen is a difference in the model rather than in post-processing.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ~3 minutes. Versions pinned to what the project targets.
!pip -q install "transformers>=4.45" accelerate qwen-vl-utils pillow
print("done")

## 1. Upload the demo payload

Locally, run:

```bash
python scripts/export_vlm_payload.py
```

That writes `vlm_payload.zip` (images + the exact prompt each one needs).
Upload it here.

In [ ]:
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()          # choose vlm_payload.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall("payload")

import json
manifest = json.load(open("payload/manifest.json"))
print(f"{len(manifest['documents'])} documents to run")
print("model:", manifest["model_id"])

## 2. Load the model

Qwen2-VL-2B in fp16 — it fits a T4 comfortably, which is why the project's
compute plan picks it as the first baseline rather than a 7B in 4-bit.

In [ ]:
import torch
from transformers import AutoProcessor

# transformers v5 renamed two things at once, and both only fail on a GPU box:
#   AutoModelForVision2Seq -> AutoModelForImageTextToText   (old name removed)
#   from_pretrained(torch_dtype=...) -> from_pretrained(dtype=...)
# Probe for each rather than pinning, so this runs on whatever Colab ships.
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM

MODEL_ID = manifest["model_id"]
try:
    model = AutoVLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
except TypeError:
    model = AutoVLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")

processor = AutoProcessor.from_pretrained(MODEL_ID)
print("loaded", MODEL_ID, "|", type(model).__name__)

## 3. Run every document

A model that returns unparseable text is a *result*, not a crash — it scores as
a missed document. So failures are recorded and the loop continues.

In [ ]:
import time, json, pathlib
from PIL import Image

SYSTEM_PROMPT = manifest["system_prompt"]
outputs = {}

for i, entry in enumerate(manifest["documents"], 1):
    image = Image.open(pathlib.Path("payload") / entry["image"]).convert("RGB")
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": entry["prompt"]},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    t0 = time.perf_counter()
    try:
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=1536, do_sample=False)
        reply = processor.decode(
            generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        error = None
    except Exception as exc:
        reply, error = "", f"{type(exc).__name__}: {exc}"

    outputs[entry["doc_id"]] = {
        "raw_response": reply,
        "elapsed_ms": round((time.perf_counter() - t0) * 1000),
        "error": error,
    }
    print(f"[{i}/{len(manifest['documents'])}] {entry['doc_id']}  "
          f"{outputs[entry['doc_id']]['elapsed_ms']/1000:.1f}s  "
          f"{'ERROR' if error else str(len(reply)) + ' chars'}")

## 4. Parse and save

`extract_json_object` is copied from the project so the parsing here matches
what the local pipeline does — models wrap JSON in prose and markdown fences,
and a fragile parser would show up later as a model-quality problem.

In [ ]:
import re, json

_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)
_TRAILING_COMMA = re.compile(r",(\s*[}\]])")

def _balanced(text):
    start = text.find("{")
    if start < 0: return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
            continue
        if ch == '"': in_str = True
        elif ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[start:i+1]
    return None

def extract_json_object(response):
    if not response or not response.strip(): return None
    cands = [response.strip()]
    m = _FENCE.search(response)
    if m: cands.append(m.group(1).strip())
    b = _balanced(response)
    if b: cands.append(b)
    for c in cands:
        for attempt in (c, _TRAILING_COMMA.sub(r"\1", c)):
            try: parsed = json.loads(attempt)
            except Exception: continue
            if isinstance(parsed, dict): return parsed
            if isinstance(parsed, list) and len(parsed) == 1 and isinstance(parsed[0], dict):
                return parsed[0]
    return None

for doc_id, result in outputs.items():
    result["parsed"] = extract_json_object(result["raw_response"])

n_ok = sum(1 for r in outputs.values() if r["parsed"] is not None)
print(f"parsed {n_ok}/{len(outputs)} responses")
for doc_id, r in outputs.items():
    if r["parsed"] is None:
        print(f"  FAILED to parse: {doc_id}  {r['raw_response'][:120]!r}")

payload = {"model": MODEL_ID.split('/')[-1], "model_id": MODEL_ID, "outputs": outputs}
with open("vlm_outputs.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=1)
print("wrote vlm_outputs.json")

In [ ]:
from google.colab import files
files.download("vlm_outputs.json")

## 5. Back on your machine

```bash
python scripts/build_demo_cache.py --vlm vlm_outputs.json
streamlit run demo/app.py
```

Screen 4 will now show the rules-vs-model comparison.